# CausalIF: bring your own data

A generic template for running the [awslabs/causalif](https://github.com/awslabs/causalif) framework on **your own** tabular dataset. Point it at a CSV, describe your factors, name the target you want to explain, and run.

CausalIF combines an LLM's background knowledge with Bayesian structure learning (Hill Climbing + BDeu, plus bootstrap stability) to move from raw observational data to a directed causal graph. The LLM is served through **Amazon Bedrock**.

**Runtime target:** AWS SageMaker Studio, or a local Jupyter kernel with AWS credentials available.

This first section just gets the environment ready: install the packages, import them, and set up a configurable AWS region.

## 1. Install packages

Install the packages directly with the notebook `%pip` magic so they land in the active kernel. SageMaker Studio images already ship `pandas`, `numpy`, `plotly`, and `pgmpy`; `causalif` reuses those and pulls in anything missing. `langchain-aws` provides the Bedrock-backed LLM (`ChatBedrockConverse`) and the Knowledge Base retriever.

In [ ]:
# CausalIF - Python packages
# SageMaker Studio images already ship pandas / numpy / plotly / pgmpy,
# but pinning causalif here keeps things reproducible across kernels.
# langchain-aws>=1.6 is required for managed Knowledge Base retrieval
# (managedSearchConfiguration in Section 5); older versions only support
# vectorSearchConfiguration for customer-managed KBs.
%pip install causalif==0.1.10 "langchain-aws>=1.6"

## 2. Import the main packages

The core CausalIF entry points we'll use:

- `set_causalif_engine` — configures the engine (LLM, dataframe, domains, etc.)
- `causalif` — runs a natural-language causal query
- `visualize_causalif_results` — renders the interactive Plotly causal graph
- `causalif_intervene` — asks interventional "what if" (do-operator) questions

`ChatBedrockConverse` (from `langchain-aws`) is the Bedrock-backed LLM CausalIF reasons with.

In [ ]:
import pandas as pd
import numpy as np

from causalif import (
    set_causalif_engine,
    causalif,
    visualize_causalif_results,
    causalif_intervene,
)
from langchain_aws import ChatBedrockConverse

print("Imports ready.")

## 3. Region and model configuration

Anything region-, model-, or dataset-specific lives in one place so it's easy to change. Defaults target **us-west-2**. Serverless Bedrock models auto-enable on first use, so there's no manual model-access step; just make sure your execution role can invoke the model (`bedrock:InvokeModel`) and isn't blocked by an IAM policy or SCP. For Anthropic Claude, a first-time user may be asked to submit brief use-case details before the first call.

For your dataset, set:

- **`DATA_PATH`** — the path to your CSV.
- **`TARGET_FACTOR`** — the exact column name you want to explain.
- **`DOMAINS`** — the field(s) your data comes from, so the LLM reasons with the right background knowledge (e.g. `["healthcare"]`, `["finance", "economics"]`).
- **`KNOWLEDGE_BASE_ID`** *(optional)* — an Amazon Bedrock Knowledge Base ID to ground reasoning in your own documents (see Section 5). Leave as `None` to use the LLM's background knowledge alone.

In [ ]:
# --- Configurable settings ---------------------------------------------------
AWS_REGION = "us-west-2"

# Bedrock model id used by CausalIF for causal reasoning.
# Swap this for any Bedrock model you have access to in AWS_REGION. Use a
# CURRENT model - older ones (e.g. Claude Sonnet 4, ...-4-20250514) are marked
# Legacy by the provider and Converse fails with 'ResourceNotFoundException:
# ... Access denied. This Model is marked by provider as Legacy'.
BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# (Optional) Amazon Bedrock Knowledge Base ID for RAG-grounded causal reasoning.
# Leave as None to run on the LLM's background knowledge alone (no retriever).
KNOWLEDGE_BASE_ID = None

# Path to YOUR observational data (CSV in the workspace root, by default).
DATA_PATH = "your-data.csv"

# The column you want to explain / find the causal drivers of.
# Must be an exact column name in your data.
TARGET_FACTOR = "<your_target_column>"

# Domain(s) your data comes from, so the LLM reasons with the right background
# knowledge, e.g. ["healthcare"], ["finance", "economics"], ["marketing"].
DOMAINS = ["<your_domain>"]
# -----------------------------------------------------------------------------

print(f"Region: {AWS_REGION}")
print(f"Model:  {BEDROCK_MODEL_ID}")
print(f"Knowledge Base ID: {KNOWLEDGE_BASE_ID or '(none - using background knowledge only)'}")
print(f"Data path: {DATA_PATH}")
print(f"Target factor: {TARGET_FACTOR}")
print(f"Domains: {DOMAINS}")

In [ ]:
# Initialise the Bedrock-backed LLM. temperature=0.0 keeps causal reasoning stable across runs.
model = ChatBedrockConverse(
    model_id=BEDROCK_MODEL_ID,
    temperature=0.0,
    region_name=AWS_REGION,
)

print("Bedrock model client created.")

## 4. Prepare the data

> ### 💡 Let Kiro do the heavy lifting
>
> This section, profiling the data, cleaning it, and writing the `factor_descriptions`, is the perfect place to **vibe code with Kiro** instead of hand-editing every cell. Load your data in the first cell below, then open Kiro chat and ask it to do the rest. Try prompts like:
>
> - *"Profile `raw_df` in this notebook: for each column show dtype, non-null count, number of unique values, a few example values, and flag anything that looks like an ID, free text, or a category code disguised as a number. Recommend which columns to exclude and why."*
> - *"Fill in `EXCLUDED_COLUMNS` and the cleaning cell for `raw_df` based on that profile: drop the identifier/free-text columns, handle missing values sensibly, and explain each choice."*
> - *"Write the `factor_descriptions` string for the columns kept in `df`, one bullet per column using the exact column names, with a plain-English meaning and units for each. Base it on the data and column names."*
> - *"Suggest a good `TARGET_FACTOR` and `DOMAINS` value for this dataset, then update Section 3."*
>
> Kiro can read the notebook, run the profiling, and edit these cells for you. Review its suggestions, then run the cells below.

Load your dataset and get it into a clean, numeric dataframe that CausalIF can analyse. The template below assumes a standard comma-separated CSV **with a header row**. Adjust the `pd.read_csv` call for your format:

- **Different delimiter** (tabs, whitespace): pass `sep="\t"` or `sep=r"\s+"`.
- **No header row**: pass `header=None` and `names=[...]` with your column names.
- **Missing-value markers** (e.g. `?`, `NA`): pass `na_values=["?", "NA"]`.

A few general rules for good causal factors:

- **Keep numeric, meaningful factors.** CausalIF reasons over the values, so columns should be numbers whose magnitude means something.
- **Drop identifiers and free text** (names, IDs, notes) — they aren't causal factors.
- **Reconsider category codes.** A numeric column that's really a category (e.g. a region code 1/2/3) can mislead a continuous analysis; either exclude it or encode it deliberately.

List any columns to leave out in `EXCLUDED_COLUMNS`.

In [ ]:
# Load your data. Adjust read_csv arguments for your file's format (see the
# notes above): sep=..., header=..., names=[...], na_values=[...].
raw_df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(raw_df)} rows, {raw_df.shape[1]} columns.")
print(f"Columns: {list(raw_df.columns)}")
raw_df.head()

In [ ]:
# Columns to exclude from the causal analysis (identifiers, free text,
# category codes, or anything not a meaningful numeric factor). Leave empty
# to keep everything.
EXCLUDED_COLUMNS = []

# Drop excluded columns, then drop rows with missing values.
df = raw_df.drop(columns=EXCLUDED_COLUMNS).dropna().reset_index(drop=True)

FACTOR_COLUMNS = list(df.columns)

# Sanity check: the target must be one of the analysis factors.
if TARGET_FACTOR not in FACTOR_COLUMNS:
    raise ValueError(
        f"TARGET_FACTOR '{TARGET_FACTOR}' is not in the analysis factors {FACTOR_COLUMNS}. "
        "Set TARGET_FACTOR (Section 3) to an exact column name, and make sure it "
        "isn't listed in EXCLUDED_COLUMNS."
    )

print(f"Analysis factors ({len(FACTOR_COLUMNS)}): {FACTOR_COLUMNS}")
print(f"Rows after cleaning: {len(df)}")
df.head()

### Factor descriptions

CausalIF reasons about causal direction using the *meaning* of each column, not just its name. Passing a `factor_descriptions` string (Markdown) with a plain-English definition of each factor gives the LLM the domain context it needs to orient edges correctly. Describe **every** factor you're analysing, one bullet per column, using your exact column names.

In [ ]:
# Describe each analysis factor in plain English. Use your exact column names.
# Replace these example lines with one bullet per column in FACTOR_COLUMNS.
factor_descriptions = """# Factor Definitions
- <column_name_1>: what this factor means and its units (and which direction is "more")
- <column_name_2>: what this factor means and its units
- <target_column>: the outcome you're explaining and its units
"""

print(factor_descriptions)

## 5. (Optional) Use a Knowledge Base retriever for RAG

CausalIF works well on the LLM's background knowledge alone, which is the default here. You can optionally ground its causal reasoning in your own domain documents through an Amazon Bedrock Knowledge Base (KB): during edge analysis, CausalIF retrieves relevant passages to inform each association vote.

To use a KB, create one in Amazon Bedrock (upload your reference documents to S3 and build a managed Knowledge Base), then paste the resulting **Knowledge Base ID** into `KNOWLEDGE_BASE_ID` in the configuration cell (Section 3).

The cell below builds the retriever **only if** `KNOWLEDGE_BASE_ID` is set. If it's `None`, the retriever is skipped and the notebook runs on background knowledge alone — no other changes needed.

In [ ]:
# Build a Knowledge Base retriever only when KNOWLEDGE_BASE_ID is set.
# retriever stays None otherwise, and Section 6 simply passes None (no RAG).
retriever = None

if KNOWLEDGE_BASE_ID:
    from langchain_aws.retrievers import AmazonKnowledgeBasesRetriever

    retriever = AmazonKnowledgeBasesRetriever(
        knowledge_base_id=KNOWLEDGE_BASE_ID,
        region_name=AWS_REGION,
        retrieval_config={"managedSearchConfiguration": {"numberOfResults": 20}},
    )
    print(f"Retriever created for Knowledge Base '{KNOWLEDGE_BASE_ID}'.")
else:
    print("No KNOWLEDGE_BASE_ID set - skipping retriever (using background knowledge only).")

## 6. Configure the CausalIF engine

`set_causalif_engine` wires everything together: the Bedrock model, your cleaned dataframe, the factor descriptions, and some analysis settings.

Key choices:

- **`domains=DOMAINS`** — gives the LLM the right background knowledge to reason with, set via `DOMAINS` in Section 3 (e.g. `["healthcare"]`, `["finance", "economics"]`). The README treats this as effectively mandatory.
- **`enable_causal_estimate=True`** — turns on causal effect estimation (ATE) so we can later ask interventional "what if" questions with `causalif_intervene`.
- **`bootstrap_iterations` / `bootstrap_threshold`** — resample the data and keep only edges whose direction is stable across resamples. These are the framework's benchmark defaults.
- **`retriever=retriever`** — the optional KB retriever from Section 5. It's `None` unless you set `KNOWLEDGE_BASE_ID`, in which case CausalIF stays on the LLM's background knowledge plus your observational data.

In [ ]:
set_causalif_engine(
    model=model,
    dataframe=df,
    domains=DOMAINS,
    factor_descriptions=factor_descriptions,
    selected_dataframe_columns=FACTOR_COLUMNS,
    retriever=retriever,
    enable_causal_estimate=True,
    bootstrap_iterations=50,
    bootstrap_threshold=0.5,
    max_parallel_queries=50,
)

print("CausalIF engine configured.")

## 7. Run the causal analysis

Now we ask the actual question. CausalIF understands a few natural-language query formats; the `what influences <target_factor>` form maps cleanly onto "what drives my target?". The target factor must be an exact column name — we use `TARGET_FACTOR` from Section 3.

This kicks off the full pipeline: LLM edge voting to build the prior, Bayesian orientation with bootstrap stability, and causal effect estimation. It makes several Bedrock calls per factor pair, so expect it to take a little while.

In [ ]:
result = causalif(f"what influences {TARGET_FACTOR}")

print(result["summary"])

## 8. Visualise the causal graph

`visualize_causalif_results` renders the discovered causal graph as an interactive Plotly figure. Nodes are coloured by their degree of separation from the target, arrows show causal direction, and hovering over edges reveals the estimated effects.

In [ ]:
fig = visualize_causalif_results(result)
fig.show()

## 9. Save the result

Write the full result dictionary to `result.json` for later inspection. The result can hold numpy values and tuples (e.g. graph edges, and dict keys keyed by factor pairs), so we convert tuple keys to strings and pass `default=str` to serialise anything else that isn't natively JSON-friendly.

In [ ]:
import json


def jsonify_keys(obj):
    """Recursively convert non-string dict keys (e.g. ('a','b')) to strings."""
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if isinstance(k, tuple):
                k = "->".join(map(str, k))          # ('a','b') -> 'a->b'
            elif not isinstance(k, (str, int, float, bool)) and k is not None:
                k = str(k)
            out[k] = jsonify_keys(v)
        return out
    if isinstance(obj, (list, tuple)):
        return [jsonify_keys(v) for v in obj]
    return obj


with open("result.json", "w", encoding="utf-8") as f:
    json.dump(jsonify_keys(result), f, indent=2, default=str)

print("Wrote result.json")